# Exercise 2 · Agent loop — DEMO & SOLUTION
**Lecturer notebook — run cell by cell while sharing the screen.**

Each **round** is one cell. The model either returns a **tool call** — the JSON schema, filled in — and your code runs the search and sends the result back; or it returns **no tool call**, which means: *final answer*.

## 0 · Setup
Run the three cells below. The last one prints a greeting from the model.

In [14]:
!pip install -q unique-sdk


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [15]:
import json
import re
import unique_sdk

# --- API key -------------------------------------------------------------------
# COURSE KEY: only for this course, revoked after Module 1 (21 Nov 2026).
# Never put real keys into a notebook — in production, keys live in a secret store.
COURSE_API_KEY = "ukey_c56fQfqC31AulM6jru-rb33KYLTkip2dloEJ5eiUkOs"

try:  # the clean way: Colab Secrets (key icon in the left sidebar), if you set one
    from google.colab import userdata
    API_KEY = userdata.get("UNIQUE_API_KEY") or COURSE_API_KEY
except Exception:
    API_KEY = COURSE_API_KEY

# --- Course environment (provided by the lecturers) ---
unique_sdk.api_base = "https://gateway.unique.app/public/chat"
unique_sdk.api_key = API_KEY
unique_sdk.app_id = "app_eyugp8qq242fzp1lle4xjqc1"

COMPANY_ID = "346686658780860534"
USER_ID = "399599116591567964"
SCOPE_IDS = ["scope_nnqbp55xfh6s25wx4mdndff0"]   # knowledge base: company annual reports
MODEL = "AZURE_GPT_51_2025_1113"

In [16]:
# Connection check: one plain LLM call
hello = unique_sdk.ChatCompletion.create(
    company_id=COMPANY_ID,
    user_id=USER_ID,
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say hello to the ETH MAS class in one short sentence."},
    ],
)
print(hello.choices[0].message.content)

Hello ETH MAS class—wishing you an inspiring and successful journey through your studies!


## The tool — what the model sees
This empty **schema** is all the model knows about the tool. Watch how it comes back *filled in* — **`reason` first**, then the query.

In [4]:
def search_knowledge_base(reason: str, search_string: str) -> str:
    """The function your code runs when the model asks for the tool.
    `reason` is not used by the search — it makes the model think before it acts, and lets you see why."""
    chunks = unique_sdk.Search.create(
        user_id=USER_ID,
        company_id=COMPANY_ID,
        searchString=search_string,
        searchType="VECTOR",
        scopeIds=SCOPE_IDS,
        limit=4,   # deliberately small: the agent has to search several times
    ).data
    return "\n\n".join(
        f"[{c.get('title') or c.get('key')}, p. {c.get('startPage')}–{c.get('endPage')}]\n{c['text']}"
        for c in chunks
    )


# The tool schema: this is ALL the model sees of your tool
search_tool = {
    "type": "function",
    "function": {
        "name": "search_knowledge_base",
        "description": "Search the internal knowledge base of annual reports (Siemens, ABB, UBS; 2021–2023). "
                       "Returns a few text passages with document title and page. "
                       "Use short, specific queries (one topic, one year, one company).",
        "parameters": {
            "type": "object",
            "properties": {
                "reason": {   # FIRST: the model writes its reasoning before it picks the query
                    "type": "string",
                    "description": "Why you search this now: what you already know and what is still missing.",
                },
                "search_string": {
                    "type": "string",
                    "description": "The search query, e.g. 'Siemens revenue fiscal year 2021'",
                },
            },
            "required": ["reason", "search_string"],
        },
    },
}

TOOLS = {"search_knowledge_base": search_knowledge_base}   # name -> Python function

In [5]:
print(json.dumps(search_tool, indent=2))

{
  "type": "function",
  "function": {
    "name": "search_knowledge_base",
    "description": "Search the internal knowledge base of annual reports (Siemens, ABB, UBS; 2021\u20132023). Returns a few text passages with document title and page. Use short, specific queries (one topic, one year, one company).",
    "parameters": {
      "type": "object",
      "properties": {
        "reason": {
          "type": "string",
          "description": "Why you search this now: what you already know and what is still missing."
        },
        "search_string": {
          "type": "string",
          "description": "The search query, e.g. 'Siemens revenue fiscal year 2021'"
        }
      },
      "required": [
        "reason",
        "search_string"
      ]
    }
  }
}


## The agent's instructions and the question

In [7]:
AGENT_SYSTEM_PROMPT = """You are a research analyst with access to a search tool over company annual reports.
Work step by step:
1. Break the question into sub-questions (per company, year and topic).
2. Search ONE sub-question per turn with a short, specific query. Read the results, then decide the next search.
   Use different queries — never repeat one.
3. Do at least 3 search rounds before you answer.
4. When you have enough evidence, write the final answer: structured, concise, and cite the
   document and page for every number, e.g. (Siemens Annual Report 2021, p. 12).
"""

QUESTION = ("In fiscal 2021 Siemens completed a large acquisition. Which company was it, "
            "what did Siemens pay, and how did that business develop in fiscal 2022 and 2023?")

# Alternative (no hop, but several searches): "Compare the 2021 and 2023 annual reports of Siemens: revenue, profit and strategic priorities."

## The loop — one cell per round
Run the next cell once, then the round cells one after another. Point out each time:
1. the model returns **no text** but `toolCalls` with the **filled-in arguments**,
2. **your code** runs the search and appends the result to the conversation,
3. in the last round: **`toolCalls = None`** → the model decided to stop.

In [8]:
messages = [
    {"role": "system", "content": AGENT_SYSTEM_PROMPT},
    {"role": "user", "content": QUESTION},
]
round_no = 0
finished = False


def next_round():
    """One loop iteration: ask the model -> tool call? run it : final answer."""
    global round_no, finished
    if finished:
        print("The agent has already finished — see the final answer above.")
        return
    round_no += 1
    print(f"ROUND {round_no} · sending {len(messages)} messages to the model …\n")

    response = unique_sdk.ChatCompletion.create(
        company_id=COMPANY_ID, user_id=USER_ID, model=MODEL, messages=messages, options={"tools": [search_tool]},
    )
    message = response.choices[0].message
    tool_calls = message.get("toolCalls") or message.get("tool_calls") or []

    if not tool_calls:
        finished = True
        messages.append({"role": "assistant", "content": message.get("content") or ""})
        print("MODEL RESPONSE:  toolCalls = None")
        print("➜ No tool call: the MODEL decided it has enough evidence. This is the final answer.\n")
        print("=" * 80)
        print(message.get("content"))
        return

    # The model did not answer — it returned the tool schema, filled in:
    print("MODEL RESPONSE:  content = " + repr(message.get("content") or None) + ",  toolCalls =")
    print(json.dumps(
        [{"id": c["id"], "type": c.get("type", "function"),
          "function": {"name": c["function"]["name"], "arguments": json.loads(c["function"]["arguments"])}}
         for c in tool_calls],
        indent=2, ensure_ascii=False))

    messages.append({"role": "assistant", "content": message.get("content") or "", "toolCalls": tool_calls})
    for call in tool_calls:
        name = call["function"]["name"]
        args = json.loads(call["function"]["arguments"])
        result = TOOLS[name](**args)                          # YOUR code executes the call
        messages.append({"role": "tool", "toolCallId": call["id"], "name": name, "content": result})
        sources = re.findall(r"^\[(.+?)\]$", result, flags=re.M)
        print(f"\nYOUR CODE ran {name}({args})")
        print(f"  → {len(sources)} passages sent back to the model: " + "; ".join(sources))

### Round 1

In [9]:
next_round()

ROUND 1 · sending 2 messages to the model …

MODEL RESPONSE:  content = None,  toolCalls =
[
  {
    "id": "call_mrmBYlhpqIjqOHU30vNE8m2t",
    "type": "function",
    "function": {
      "name": "search_knowledge_base",
      "arguments": {
        "reason": "Identify major acquisition Siemens completed in fiscal 2021",
        "search_string": "Siemens major acquisition fiscal 2021 consideration"
      }
    }
  }
]

YOUR CODE ran search_knowledge_base({'reason': 'Identify major acquisition Siemens completed in fiscal 2021', 'search_string': 'Siemens major acquisition fiscal 2021 consideration'})
  → 4 passages sent back to the model: Siemens_2022.pdf, p. 51–51; Siemens_2022.pdf, p. 51–51; Siemens_2021.pdf, p. 53–53; Siemens_2021.pdf, p. 24–25


### Round 2 — note: the query now uses what round 1 found

In [10]:
next_round()

ROUND 2 · sending 4 messages to the model …

MODEL RESPONSE:  content = None,  toolCalls =
[
  {
    "id": "call_Kajk0LSfuqy6EkQOHKBa40sd",
    "type": "function",
    "function": {
      "name": "search_knowledge_base",
      "arguments": {
        "reason": "See overview of Siemens acquisitions in 2021 to identify which is described as large",
        "search_string": "Siemens 2021 major acquisition Varian Medical Systems purchase price"
      }
    }
  }
]

YOUR CODE ran search_knowledge_base({'reason': 'See overview of Siemens acquisitions in 2021 to identify which is described as large', 'search_string': 'Siemens 2021 major acquisition Varian Medical Systems purchase price'})
  → 4 passages sent back to the model: Siemens_2021.pdf, p. 52–53; Siemens_2022.pdf, p. 51–51; Siemens_2021.pdf, p. 53–53; Siemens_2021.pdf, p. 105–106


### Round 3

In [12]:
next_round()

ROUND 4 · sending 8 messages to the model …

MODEL RESPONSE:  content = None,  toolCalls =
[
  {
    "id": "call_l65CiONOV4cnzivj3uqvocvA",
    "type": "function",
    "function": {
      "name": "search_knowledge_base",
      "arguments": {
        "reason": "Look for qualitative description of Varian integration and performance in 2022 and 2023",
        "search_string": "Siemens 2023 Varian integration synergies oncology-as-a-service"
      }
    }
  }
]

YOUR CODE ran search_knowledge_base({'reason': 'Look for qualitative description of Varian integration and performance in 2022 and 2023', 'search_string': 'Siemens 2023 Varian integration synergies oncology-as-a-service'})
  → 4 passages sent back to the model: Siemens_2021.pdf, p. 52–53; Siemens_2022.pdf, p. 13–14; Siemens_2023.pdf, p. 13–13; Siemens_2022.pdf, p. 13–13


### Round 4

In [13]:
next_round()

ROUND 5 · sending 10 messages to the model …

MODEL RESPONSE:  toolCalls = None
➜ No tool call: the MODEL decided it has enough evidence. This is the final answer.

**Which company and what Siemens paid (fiscal 2021)**  

In fiscal 2021, Siemens completed the large acquisition of **Varian Medical Systems, Inc. (Varian)**, a U.S.-based specialist in cancer care and radiation oncology solutions. Varian was integrated into **Siemens Healthineers**.

- **Purchase price:** Siemens paid **US$16.4 billion in cash**, corresponding to **€13.9 billion** at the acquisition date.  
  (Siemens Annual Report 2021, Note 3, p. 52–53)

The preliminary purchase price allocation included goodwill of **€8.0 billion** and other intangible assets of **€6.3 billion**, mainly related to oncology technologies, customer relationships, and order backlog. (Siemens Annual Report 2021, Note 3, p. 52–53)

Varian contributed **€1,241 million in revenue** and a **net loss of €50 million** from the acquisition date (Ap

### Round 5 (only if the agent is still searching — if needed, run `next_round()` again)

In [ ]:
next_round()

## What the model saw
The agent has no hidden memory: every round re-sends this growing list.

In [ ]:
# The whole conversation the model saw — the agent's "memory" is just this list
for i, m in enumerate(messages, 1):
    role = m["role"]
    if role == "assistant" and m.get("toolCalls"):
        calls = ", ".join(f'{c["function"]["name"]}({c["function"]["arguments"]})' for c in m["toolCalls"])
        text = "→ tool call: " + calls
    elif role == "tool":
        n = len(re.findall(r"^\[", m["content"], flags=re.M))
        text = f"{n} passages ({len(m['content']):,} chars)"
    elif role == "assistant":
        text = f"FINAL ANSWER ({len(m['content']):,} chars)"
    else:
        text = m["content"].strip().splitlines()[0][:90]
    print(f"{i:>2}  {role:<9}  {text}")

---
# Solution: the same loop as one function
This is what the students write in Exercise 2.

In [ ]:
def run_agent(question: str, max_steps: int = 10, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        response = unique_sdk.ChatCompletion.create(
            company_id=COMPANY_ID, user_id=USER_ID, model=MODEL, messages=messages, options={"tools": [search_tool]},
        )
        message = response.choices[0].message
        tool_calls = message.get("toolCalls") or message.get("tool_calls") or []

        if not tool_calls:                                   # no tool call = the MODEL decided: final answer
            if verbose:
                print(f"Step {step} · no tool call → final answer\n" + "=" * 60)
            return message.get("content") or ""

        messages.append({"role": "assistant", "content": message.get("content") or "", "toolCalls": tool_calls})
        for call in tool_calls:
            name = call["function"]["name"]
            args = json.loads(call["function"]["arguments"])
            result = TOOLS[name](**args)                     # the model proposes, YOUR code executes
            if verbose:
                print(f"Step {step} · {name}({args})")
            messages.append({"role": "tool", "toolCallId": call["id"], "name": name, "content": result})

    return "Stopped: max_steps reached without a final answer."   # safety limit, not a decision

In [ ]:
answer = run_agent(QUESTION)
print(answer)

### Discussion
- **Who decided to stop?** The model — by not calling a tool. `max_steps` is only a safety limit.
- **Non-determinism:** run it again — the queries (and sometimes the result) differ. In our tests 4 of 5 runs found Varian; once the agent picked Supplyframe and researched the wrong company, with correct citations. One run tells you little: test over many runs.
- **Cost:** every round re-sends the whole history — see how the list above grows.